# Binary semi-synthetic study · walmart
Same algorithms, quality measure (M2), constraints and task design on M5, Walmart and Favorita.

- Descriptors: k ∈ {20, 50, 100, 400} binary attributes; attribute j is 1 for a share p_j ~ U(0.5, 0.9)
  of the records (uneven binary, so deep descriptions keep enough records).
- True subgroup: 2, 4, 6, 8 or 10 conditions, 1–15% coverage, ≥ 2 stores; level step δ = snr × σ
  (σ = null spread of random descriptions of that depth), snr 8 / 16, outliers 0 / 20%, 10 repetitions.
- Methods: BS w=10, BS w=50, BUS, BUS +local, BUS-α, BUS→BS pool, BUS snn, BUS snn +local, BUS→BS pool snn.
- Bart's BUS (closest / α pairs) runs exactly, with the time BS w=50 took on the same task (at least 60 s); unfinished runs are reported.
Resumable: progress in results_binary/<dataset>/progress.txt; rerunning skips finished tasks.

In [ ]:
import os
import time
import heapq
import itertools
from collections import Counter

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 110)
N_JOBS = os.cpu_count()
print("CPU cores:", N_JOBS)

In [ ]:
DATASET = "walmart"
QUICK = False          # True = tiny grids, only to test the notebook end to end
UNIT = "$"
PERIOD = "week"
PERIODS_PER_YEAR = 52

# ---- Search parameters (Bart §6.3.1) ----
D_MAX = 3                # max conditions in a description
Q = 20                   # result set size
W = 50                   # default beam width
U = 200                  # mini-clusters (pairs) for every BUS variant and both ensembles
POOL_SIZE = 30           # pool ensemble: conditions the BS step may use
W_MC = 5                 # per-mini-cluster ensemble: beam width of each small beam search

# ---- Narrow findings ----
MIN_COV = 0.02
MAX_COV = 0.15
MIN_STORES = 2
MIN_PEERS = 5

# ---- Reference point ----
BASE_LEN = 52             # first 52 weeks
MIN_BASE_LEVEL = 500      # $ per week

# ---- Experiment 0: real data ----
MEASURES = ["M1", "M2", "M3"]
METHODS = ["BS", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]
NU = 20 if QUICK else 500                     # permutation runs for the DFD (Bart used 500)

# ---- Experiment 1: planted subgroups ----
PATTERNS = {"level": "M2", "trend": "M2", "timing": "M3"}    # pattern -> the measure that should catch it
SNR = [2, 8] if QUICK else [1, 2, 4, 8]                      # planted effect in units of the null spread sigma (see E1)
LITS = [2] if QUICK else [2, 3]                               # conditions in the planted description
OUTLIERS = [0.0] if QUICK else [0.0, 0.2]                     # share of planted records left unchanged
REPS = 1 if QUICK else 2                                      # different planted descriptions per setting
PLANT_COVERAGE = (0.02, 0.08)
E1_METHODS = ["BS", "BS w=10", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]

# ---- Experiment 2: interaction-only subgroups ----
E2_SNR = [8] if QUICK else [4, 8]
E2_REPS = 1 if QUICK else 3
E2_METHODS = ["BS w=5", "BS w=10", "BS w=20", "BS", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]

HIT_JACCARD = 0.8        # a planted subgroup counts as found if a subgroup with Jaccard >= 0.8 was evaluated
SEED = 0
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
RESULTS_DIR = "/kaggle/working/results_v3" if ON_KAGGLE else "results_v3"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ================= v4 experiment settings =================
# pattern -> the quality measure that matches it
PATTERNS_V4 = {"level": "M2", "trend": "M2", "stockout": "M2",      # money patterns (Konijn-style excess)
               "season": "M3", "lag": "M3",                           # shape patterns (Bart phi_cluster)
               "volatility": "M4"}                                    # variance pattern (new M4)
SNR_V4 = [2, 8] if QUICK else [2, 4, 8]           # planted strength in units of the null spread sigma
LAG_STEPS = {2: 1, 4: 2, 8: 4}                    # lag pattern: shift by this many LAG_UNITs instead of snr x sigma
LAG_UNIT = {"day": 7, "week": 1, "month": 1}[PERIOD]
LITS_V4 = [2] if QUICK else [2, 3]
REPS_V4 = 1 if QUICK else 3

# seeds (mini-clusters)
U_SEEDS = 200          # pairs per pair-based seed type
KNN = 10               # neighbours for the kNN / shared-nearest-neighbour graph
SNN_MIN = 5            # edge in the SNN graph if two mutual neighbours share >= SNN_MIN neighbours
Z_COMP_MIN = 3         # smallest non-convex component used as a mini-cluster
MAJORITY = 0.6         # a component describes itself by conditions shared by >= 60% of its members
N_BLOCKS = 36          # time blocks of the denoised representation
LOCAL_TOPK = 10        # Q3: local search starts from BUS's 10 best candidates

SEED_TYPES = ["closest-$", "closest-rel", "closest-blocks", "snn-pairs", "snn-components"]
METHODS_V4 = ["BS w=10", "BS",
              "BUS closest-$", "BUS→BS pool α-$",                       # the old BUS and old pool ensemble
              "BUS closest-rel", "BUS closest-blocks", "BUS snn-pairs", "BUS snn-components",
              "BUS→BS pool blocks", "BUS→BS pool snn",
              "BUS snn-pairs +local", "BUS closest-blocks +local"]
RESULTS_DIR = "/kaggle/working/results_v4" if ON_KAGGLE else "results_v4"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ================= v5 settings =================
K_ATTRS = [100, 400]
LITS_V5 = [3, 4] if QUICK else [3, 4, 5, 6]
SNR_V5 = [8] if QUICK else [8, 16]
OUTLIERS_V5 = [0.2] if QUICK else [0.0, 0.2]
REPS_V5 = 1 if QUICK else 15
MIN_COV_V5 = 0.01
TOP_ENRICHED = 8            # conditions per mini-cluster used to build descriptions
METHODS_V5 = ["BS w=10", "BS w=50", "BUS snn-comp", "BUS snn-comp +local", "BUS→BS pool snn-comp"]
RESULTS_DIR = "results_v5"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ================= v6 settings (override v5) =================
MIN_SUPPORT = 2            # an itemset must be contained in >= 2 mini-cluster baskets
GUIDED_WIDTH = 5           # beam width of the guided local search
GUIDED_ADD_TOP = 20        # best-supported co-occurring conditions tried per step
METHOD_SET = "v5"          # "v5" = the planned v5 methods; "v6" = also the itemset methods
METHODS_V5 = ["BS w=10", "BS w=50", "BUS snn-comp", "BUS snn-comp +local", "BUS→BS pool snn-comp"]
if METHOD_SET == "v6":
    METHODS_V5 += ["BUS itemsets", "BUS itemsets +guided beam"]
RESULTS_DIR = "results_v6" if METHOD_SET == "v6" else "results_v5"
N_JOBS = min(N_JOBS, 7)    # leave memory headroom
os.makedirs(RESULTS_DIR, exist_ok=True)
BATCH = 2 * N_JOBS

In [ ]:
final = pd.read_csv("final_walmart_data.csv", usecols=["Store", "Dept", "Date"], parse_dates=["Date"])
record_index = final[["Store", "Dept"]].drop_duplicates().sort_values(["Store", "Dept"])
dates = pd.DatetimeIndex(sorted(final["Date"].unique()))
train = pd.read_csv("Walmart/train.csv", parse_dates=["Date"])
sales = (train[train["Weekly_Sales"] >= 0]
         .pivot(index=["Store", "Dept"], columns="Date", values="Weekly_Sales")
         .reindex(index=pd.MultiIndex.from_frame(record_index), columns=dates)
         .interpolate(axis=1, limit_area="inside"))
assert sales.notna().all().all()
S = sales.to_numpy(dtype=float)
records = record_index.rename(columns={"Store": "store", "Dept": "item"}).reset_index(drop=True)

features = pd.read_csv("Walmart/features.csv", parse_dates=["Date"])
features = features[features["Date"].between(dates[0], dates[-1])].sort_values(["Store", "Date"])
stores = pd.read_csv("Walmart/stores.csv")


def summarise_series(values, per_year):
    y = values.to_numpy(dtype=float)
    t = np.arange(len(y))
    slope, intercept = np.polyfit(t, y, 1)
    return pd.Series({"mean": y.mean(), "slope": slope * per_year, "volatility": (y - (slope * t + intercept)).std()})


store_desc = []
for variable in ["Temperature", "Fuel_Price", "CPI", "Unemployment"]:
    summary = features.groupby("Store")[variable].apply(summarise_series, per_year=52).unstack()
    summary.columns = [f"{variable}_{c}" for c in summary.columns]
    store_desc.append(summary)
store_desc = pd.concat(store_desc, axis=1)
md_cols = [f"MarkDown{i}" for i in range(1, 6)]
md_period = features[features["Date"] >= features.loc[features[md_cols].notna().any(axis=1), "Date"].min()]
store_desc = store_desc.join(stores.set_index("Store")[["Type", "Size"]])
store_desc["MarkDown_per_sqft"] = md_period[md_cols].fillna(0).sum(axis=1).groupby(md_period["Store"]).mean() / store_desc["Size"]
records = records.merge(store_desc, left_on="store", right_index=True, how="left")

categorical_desc = [("Type", "store")]
numeric_desc = [(c, None, "store") for c in store_desc.columns if c != "Type"]
print(S.shape, records.shape)

In [ ]:
level = S[:, :BASE_LEN].mean(axis=1)
item = records["item"].astype(str).to_numpy()

ok = level >= MIN_BASE_LEVEL
ok &= pd.Series(ok).groupby(item).transform("sum").to_numpy() >= MIN_PEERS
records = records[ok].reset_index(drop=True)
S, level, item = S[ok], level[ok], item[ok]
N, T = S.shape

group_sales = pd.DataFrame(S).groupby(item).transform("sum").to_numpy()
group_level = pd.Series(level).groupby(item).transform("sum").to_numpy()
peer_index = (group_sales - S) / (group_level - level)[:, None]
expected = level[:, None] * peer_index
excess = S - expected
rel_dev = np.clip(excess / level[:, None], -3, 3)
store_codes = pd.factorize(records["store"])[0]
del group_sales, peer_index

print(f"{DATASET}: N = {N} records, T = {T} {PERIOD}s, {records['store'].nunique()} stores, "
      f"{records['item'].nunique()} items, base window = {BASE_LEN} {PERIOD}s")
print(f"Base-window excess per record (0 by construction): max |.| = {np.abs(excess[:, :BASE_LEN].sum(axis=1)).max():.6f}")

In [ ]:
LABELS = ["very_low", "low", "medium", "high", "very_high"]


def z_bins(z):
    return pd.cut(z, bins=[-np.inf, -1.5, -0.5, 0.5, 1.5, np.inf], labels=LABELS).astype(str)


descriptor_cols, desc_level = [], {}
for column, level_ in categorical_desc:
    records[column] = records[column].astype(str)
    descriptor_cols.append(column)
    desc_level[column] = level_
for column, zgroup, level_ in numeric_desc:
    values = records[column].astype(float)
    if zgroup is None:
        z = (values - values.mean()) / values.std()
        name = f"{column}_cat"
    else:
        grouped = values.groupby(records[zgroup])
        z = (values - grouped.transform("mean")) / grouped.transform("std").replace(0, np.nan)
        name = f"{column}_vs_{zgroup}_cat"
    records[name] = z_bins(z.fillna(0))
    descriptor_cols.append(name)
    desc_level[name] = level_

A = records[descriptor_cols].astype(str).to_numpy(dtype=object)


def describe(desc):
    return " AND ".join(f"{descriptor_cols[j]} = {v}" for j, v in desc)


print(f"{len(descriptor_cols)} descriptors, {sum(records[c].nunique() for c in descriptor_cols)} conditions")
pd.DataFrame({"level": desc_level, "distinct values": records[descriptor_cols].nunique()})

In [ ]:
import hashlib
class QualityBase:
    def __init__(self, N, store_codes, min_cov=None, max_cov=None, min_stores=None):
        min_cov = MIN_COV if min_cov is None else min_cov
        max_cov = MAX_COV if max_cov is None else max_cov
        self.N = N
        self.store_codes = store_codes
        self.n_store_codes = int(store_codes.max()) + 1
        self.min_n = max(2, int(np.ceil(min_cov * N)))
        self.max_n = int(np.floor(max_cov * N))
        self.min_stores = MIN_STORES if min_stores is None else min_stores
        self.cache = {}
        self.track = None        # planted mask, if the evaluations should be traced
        self.trace = []          # (time, quality, jaccard with planted, valid)

    def n_stores(self, mask):
        return np.count_nonzero(np.bincount(self.store_codes[mask], minlength=self.n_store_codes))

    def expandable(self, mask):
        return mask.sum() >= self.min_n and self.n_stores(mask) >= self.min_stores

    def satisfies_constraints(self, mask):
        n = mask.sum()
        return n <= self.max_n and n < self.N and self.expandable(mask)

    def optimistic_estimate(self, mask):
        return None

    def __call__(self, mask):
        key = hashlib.blake2b(mask.tobytes(), digest_size=16).digest()   # 16 bytes instead of N
        if key not in self.cache:
            self.cache[key] = self.compute(mask)
            if self.track is not None:
                inter = np.count_nonzero(mask & self.track)
                union = np.count_nonzero(mask | self.track)
                self.trace.append((time.time(), self.cache[key], inter / union, self.satisfies_constraints(mask)))
        return self.cache[key]


class MonetaryQuality(QualityBase):
    def __init__(self, excess, level, store_codes, relative=True, **constraints):
        super().__init__(len(level), store_codes, **constraints)
        self.relative = relative
        self.x = excess.sum(axis=1, dtype=np.float64)
        self.level = level
        self.L_T = level.sum()
        self.y = self.x - level * self.x.sum() / self.L_T

    def signed(self, mask):
        if not self.relative:
            return self.x[mask].sum()
        return self.y[mask].sum() * self.L_T / (self.L_T - self.level[mask].sum())

    def compute(self, mask):
        return abs(self.signed(mask))

    def optimistic_estimate(self, mask):
        v = self.y[mask] if self.relative else self.x[mask]
        bound = max(v[v > 0].sum(), -v[v < 0].sum())
        if self.relative:
            bound *= self.L_T / (self.L_T - self.level[mask].sum())
        return bound


class ClusterQuality(QualityBase):
    def __init__(self, D, store_codes, **constraints):
        super().__init__(len(D), store_codes, **constraints)
        self.D = D
        self.rowsum = D.sum(axis=1, dtype=np.float64)

    def components(self, mask):
        idx = np.flatnonzero(mask)
        n, n_c = len(idx), self.N - len(idx)
        within = self.D[np.ix_(idx, idx)].sum(dtype=np.float64)
        between = self.rowsum[idx].sum() - within
        return between / (n * n_c), within / (n * (n - 1))

    def compute(self, mask):
        alpha, beta = self.components(mask)
        return alpha / (beta + 1)


def _cityblock_block(Xb, X):
    return cdist(Xb, X, metric="cityblock")


def pairwise_cityblock(X, block=512):
    """Full N x N city-block distance matrix, row blocks computed in parallel."""
    N_ = len(X)
    dtype = np.float32 if N_ > 6000 else np.float64
    starts = list(range(0, N_, block))
    parts = Parallel(n_jobs=N_JOBS)(delayed(_cityblock_block)(X[s:s + block], X) for s in starts)
    D = np.empty((N_, N_), dtype=dtype)
    for s, part in zip(starts, parts):
        D[s:s + len(part)] = part
    return D


def update_rows(D, X, rows):
    """Distances after the target rows `rows` changed: only those rows / columns are recomputed."""
    D = D.copy()
    if len(rows):
        new = cdist(X[rows], X, metric="cityblock").astype(D.dtype)
        D[rows, :] = new
        D[:, rows] = new.T
    return D




In [ ]:
def mask_of(A, desc):
    mask = np.ones(len(A), dtype=bool)
    for j, v in desc:
        mask &= A[:, j] == v
    return mask


class ResultSet:
    def __init__(self, q):
        self.q = q
        self.items = {}

    def insert(self, quality, desc, mask):
        key = mask.tobytes()
        old = self.items.get(key)
        if old is not None and len(old[1]) <= len(desc):
            return
        self.items[key] = (quality, desc, mask)
        if len(self.items) > self.q:
            del self.items[min(self.items, key=lambda k: self.items[k][0])]

    def full(self):
        return len(self.items) >= self.q

    def threshold(self):
        return min(v[0] for v in self.items.values()) if self.full() else -np.inf

    def sorted(self):
        return sorted(self.items.values(), key=lambda x: -x[0])


def oe_below_threshold(phi, mask, result_set):
    if not result_set.full():
        return False
    oe = phi.optimistic_estimate(mask)
    return oe is not None and oe < result_set.threshold()


def refine(seed, seed_mask, A, pool=None):
    used = {j for j, _ in seed}
    if pool is None:
        options = ((j, v) for j in range(A.shape[1]) if j not in used for v in pd.unique(A[seed_mask, j]))
    else:
        options = ((j, v) for j, v in pool if j not in used and (A[seed_mask, j] == v).any())
    for condition in options:
        yield tuple(sorted(seed + (condition,), key=lambda c: c[0]))


def _beam(A, phi, pool, w, d, prune, result_set, stats, evaluated_keys, mask_cache):
    def get_mask(desc):
        if desc not in mask_cache:
            mask_cache[desc] = mask_of(A, desc)
        return mask_cache[desc]

    candidate_queue = [()]
    seen, tie = set(), itertools.count()
    for level_ in range(1, d + 1):
        beam = []
        for seed in candidate_queue:
            seed_mask = get_mask(seed)
            if prune and seed and oe_below_threshold(phi, seed_mask, result_set):
                stats["pruned_oe_seed"] += 1
                continue
            for desc in refine(seed, seed_mask, A, pool):
                if desc in seen:
                    continue
                seen.add(desc)
                stats["generated"] += 1
                mask = mask_of(A, desc)          # not cached: only seeds are reused
                if not phi.expandable(mask):
                    stats["skipped_coverage"] += 1
                    continue
                if prune and seed and mask.sum() == seed_mask.sum():
                    stats["pruned_redundant"] += 1
                    continue
                quality = phi(mask)
                stats["visits"] += 1
                evaluated_keys.add(desc)
                if phi.satisfies_constraints(mask):
                    result_set.insert(quality, desc, mask)
                if prune and oe_below_threshold(phi, mask, result_set):
                    stats["pruned_oe"] += 1
                    continue
                item_ = (quality, next(tie), desc)
                if len(beam) < w:
                    heapq.heappush(beam, item_)
                else:
                    heapq.heappushpop(beam, item_)
        candidate_queue = [desc for _, _, desc in sorted(beam, reverse=True)]


def beam_search(A, phi, w=W, d=D_MAX, q=Q, prune=False, pool=None):
    stats, result_set, keys = Counter(), ResultSet(q), set()
    _beam(A, phi, pool, w, d, prune, result_set, stats, keys, {})
    stats["evaluated"] = len(keys)
    return result_set.sorted(), stats


def top_pairs(D, u=U, kind="alpha", block=512):
    """Block-wise pair selection over the lower triangle.
    kind="closest": the u smallest distances (plain BUS, Alg. 2).
    kind="alpha":   the u largest alpha_ij / (D_ij + 1) (alpha-matrix, §4.3.2)."""
    N_ = len(D)
    rowsum = D.sum(axis=1, dtype=np.float64)
    scores, rows_, cols_ = [], [], []
    for s in range(0, N_, block):
        Db = D[s:s + block].astype(np.float64)
        i = np.arange(s, s + len(Db))[:, None]
        j = np.arange(N_)[None, :]
        if kind == "closest":
            score = -Db
        else:
            alpha = (rowsum[i] + rowsum[j] - 2 * Db) / (2 * (N_ - 2))
            score = alpha / (Db + 1)
        score = np.where(j < i, score, -np.inf).ravel()
        k = min(u, int(np.isfinite(score).sum()))
        if k == 0:
            continue
        best = np.argpartition(-score, k - 1)[:k]
        scores.append(score[best])
        rows_.append(s + best // N_)
        cols_.append(best % N_)
    scores, rows_, cols_ = np.concatenate(scores), np.concatenate(rows_), np.concatenate(cols_)
    order = np.argsort(-scores)[:u]
    return [(int(rows_[o]), int(cols_[o])) for o in order]


def shared_conditions(mc, A):
    first = mc[0]
    return [(j, A[first, j]) for j in range(A.shape[1]) if all(A[i, j] == A[first, j] for i in mc)]


def evaluate_candidates(A, phi, candidates, q=Q, prune=False):
    stats = Counter(generated=len(candidates))
    result_set = ResultSet(q)
    dead = set()
    for desc in sorted(candidates, key=len):
        if prune and len(desc) > 1 and any(
            sub in dead for size in range(1, len(desc)) for sub in itertools.combinations(desc, size)
        ):
            stats["pruned_superset"] += 1
            continue
        mask = mask_of(A, desc)
        if not phi.expandable(mask):
            stats["skipped_coverage"] += 1
            dead.add(desc)
            continue
        quality = phi(mask)
        stats["evaluated"] += 1
        stats["visits"] += 1
        if phi.satisfies_constraints(mask):
            result_set.insert(quality, desc, mask)
        if prune and oe_below_threshold(phi, mask, result_set):
            dead.add(desc)
    return result_set.sorted(), stats


def bus(A, phi, pairs, d=D_MAX, q=Q, prune=False):
    candidates = set()
    for mc in pairs:
        conditions = shared_conditions(mc, A)
        for size in range(1, d + 1):
            candidates.update(itertools.combinations(conditions, size))
    top, stats = evaluate_candidates(A, phi, candidates, q, prune)
    stats["mini_clusters"] = len(pairs)
    return top, stats


def ensemble_pool(A, phi, pairs, pool_size=POOL_SIZE, prune=False):
    votes = Counter(c for mc in pairs for c in shared_conditions(mc, A))
    pool = [c for c, _ in votes.most_common(pool_size)]
    top, stats = beam_search(A, phi, prune=prune, pool=pool)
    stats["mini_clusters"] = len(pairs)
    stats["pool"] = len(pool)
    return top, stats, pool


def ensemble_per_mc(A, phi, pairs, w=W_MC, d=D_MAX, q=Q, prune=False):
    stats, result_set, keys, mask_cache = Counter(), ResultSet(q), set(), {}
    spaces = set()
    for mc in pairs:
        space = tuple(shared_conditions(mc, A))
        if not space:
            stats["mc_empty"] += 1
            continue
        if space in spaces:
            stats["mc_duplicate_space"] += 1
            continue
        spaces.add(space)
        _beam(A, phi, list(space), w, d, prune, result_set, stats, keys, mask_cache)
    stats["evaluated"] = len(keys)
    stats["mini_clusters"] = len(pairs)
    stats["mc_spaces"] = len(spaces)
    union = sorted({c for space in spaces for c in space}, key=lambda c: (c[0], str(c[1])))
    return result_set.sorted(), stats, union


def run_method(method, A, phi, D, prune, pairs=None):
    """method: "BS", "BS w=<k>", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC".
    Runtime includes mini-cluster selection unless precomputed pairs are passed."""
    start = time.time()
    pool = None
    if method.startswith("BS"):
        width = int(method.split("w=")[1]) if "w=" in method else W
        top, stats = beam_search(A, phi, w=width, prune=prune)
    else:
        if pairs is None:
            pairs = top_pairs(D, kind="closest" if method == "BUS" else "alpha")
        stats_pair = time.time() - start
        if method in ("BUS", "BUS-α"):
            top, stats = bus(A, phi, pairs, prune=prune)
        elif method == "BUS→BS pool":
            top, stats, pool = ensemble_pool(A, phi, pairs, prune=prune)
        else:
            top, stats, pool = ensemble_per_mc(A, phi, pairs, prune=prune)
        stats["pair_time"] = stats_pair
    stats["runtime"] = time.time() - start
    return top, stats, pool


def top_keys(top):
    return {m.tobytes() for _, _, m in top}


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 1.0

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

expected = S - excess
POST = np.arange(T) >= BASE_LEN
SIN = np.sin(2 * np.pi * np.arange(T) / PERIODS_PER_YEAR)


class VolatilityQuality(QualityBase):
    """M4: how much more (or less) the subgroup's deviation from its peers fluctuates after the base window.
    v_i = log(var post / var base) of r_i(t); quality = |mean v in S - mean v in complement| * sqrt(n n_c / N)."""

    def __init__(self, rel, store_codes, **constraints):
        super().__init__(len(rel), store_codes, **constraints)
        self.v = np.log((rel[:, BASE_LEN:].var(axis=1) + 1e-6) / (rel[:, :BASE_LEN].var(axis=1) + 1e-6))
        self.total = self.v.sum()

    def compute(self, mask):
        n = mask.sum()
        s = self.v[mask].sum()
        return abs(s / n - (self.total - s) / (self.N - n)) * np.sqrt(n * (self.N - n) / self.N)


def block_means(rel):
    post = rel[:, BASE_LEN:]
    k = max(1, post.shape[1] // N_BLOCKS)
    m = (post.shape[1] // k) * k
    return post[:, :m].reshape(len(rel), -1, k).mean(axis=2)


def apply_pattern(kind, rows, strength, rng, excess, expected, level):
    """Returns the planted excess. strength: share of normal sales (level/trend/season), share of post-base
    periods zeroed (stockout), extra fluctuation (volatility) or periods of lag (lag)."""
    exc = excess.copy()
    if kind == "level":
        exc[rows] += strength * POST[None, :] * level[rows, None]
    elif kind == "trend":
        ramp = np.clip((np.arange(T) - BASE_LEN + 1) / (T - BASE_LEN), 0, None) * POST
        exc[rows] += 2 * strength * ramp[None, :] * level[rows, None]
    elif kind == "season":
        exc[rows] += strength * SIN[None, :] * level[rows, None]
    elif kind == "stockout":
        actual = exc[rows] + expected[rows]
        lost = (rng.random(actual.shape) < strength) & POST[None, :]
        exc[rows] = np.where(lost, 0.0, actual) - expected[rows]
    elif kind == "lag":
        actual = exc[rows] + expected[rows]
        k = int(strength)
        shifted = np.concatenate([np.repeat(actual[:, :1], k, axis=1), actual[:, :-k]], axis=1)
        exc[rows] = shifted - expected[rows]
    elif kind == "volatility":
        post = exc[rows][:, POST]
        mean = post.mean(axis=1, keepdims=True)
        block = exc[rows]
        block[:, POST] = mean + (1 + strength) * (post - mean)
        exc[rows] = block
    return exc


# ---------------- seeds (mini-clusters)
def knn_lists(D, k=KNN, block=1024):
    out = np.empty((len(D), k), dtype=np.int64)
    for s in range(0, len(D), block):
        Db = np.array(D[s:s + block], dtype=np.float32)
        Db[np.arange(len(Db)), np.arange(s, s + len(Db))] = np.inf
        part = np.argpartition(Db, k, axis=1)[:, :k]
        out[s:s + len(Db)] = part
    return out


def snn_edges(D):
    knn = knn_lists(D)
    sets = [set(r) for r in knn.tolist()]
    edges = []
    for i, row in enumerate(knn.tolist()):
        for j in row:
            if j > i and i in sets[j]:
                edges.append((len(sets[i] & sets[j]), float(D[i, j]), i, j))
    return edges


def seeds_snn_pairs(D, u=U_SEEDS):
    edges = sorted(snn_edges(D), key=lambda e: (-e[0], e[1]))
    return [(i, j) for _, _, i, j in edges[:u]]


def seeds_snn_components(D, u=U_SEEDS):
    edges = [e for e in snn_edges(D) if e[0] >= SNN_MIN]
    if not edges:
        return []
    i = np.array([e[2] for e in edges])
    j = np.array([e[3] for e in edges])
    graph = coo_matrix((np.ones(len(i)), (i, j)), shape=(len(D), len(D)))
    _, labels = connected_components(graph, directed=False)
    comps = pd.Series(np.arange(len(D))).groupby(labels).apply(list)
    max_size = max(50, int(0.05 * len(D)))
    comps = [tuple(c) for c in comps if Z_COMP_MIN <= len(c) <= max_size]
    return sorted(comps, key=len, reverse=True)[:u]


def conditions_of(mc, A_, rule):
    if rule == "all":
        return shared_conditions(mc, A_)
    sub = A_[list(mc)]
    conds = []
    for j in range(A_.shape[1]):
        values, counts = np.unique(sub[:, j].astype(str), return_counts=True)
        best = counts.argmax()
        if counts[best] >= MAJORITY * len(mc):
            conds.append((j, values[best]))
    return conds


def candidates_of(mcs, A_, rule):
    cands = set()
    for mc in mcs:
        conds = conditions_of(mc, A_, rule)
        for size in range(1, D_MAX + 1):
            cands.update(itertools.combinations(conds, size))
    return cands


def pool_of(mcs, A_, rule, pool_size=POOL_SIZE):
    votes = Counter(c for mc in mcs for c in conditions_of(mc, A_, rule))
    return [c for c, _ in votes.most_common(pool_size)]


# ---------------- Q3: local search from BUS candidates
def local_search(A_, phi, starts, result_set, stats, max_rounds=10):
    """Hill climbing in the lattice from the best BUS candidates: drop one condition, add one condition
    (values present in the subgroup), or swap the value of one condition; move while quality improves."""
    frontier = [desc for _, desc, _ in starts[:LOCAL_TOPK]]
    seen = set(frontier)
    for _ in range(max_rounds):
        nxt = []
        for desc in frontier:
            mask = mask_of(A_, desc)
            best_q, best_d = phi(mask) if phi.expandable(mask) else -np.inf, None
            neighbours = []
            if len(desc) > 1:
                neighbours += [tuple(c for c in desc if c != drop) for drop in desc]
            used = {j for j, _ in desc}
            if len(desc) < D_MAX:
                for j in range(A_.shape[1]):
                    if j not in used:
                        neighbours += [tuple(sorted(desc + ((j, v),), key=lambda c: c[0])) for v in pd.unique(A_[mask, j])]
            for c in desc:
                rest = tuple(x for x in desc if x != c)
                for v in pd.unique(A_[mask_of(A_, rest), c[0]]):
                    if v != c[1]:
                        neighbours.append(tuple(sorted(rest + ((c[0], v),), key=lambda x: x[0])))
            for nb in neighbours:
                if nb in seen:
                    continue
                seen.add(nb)
                m = mask_of(A_, nb)
                if not phi.expandable(m):
                    continue
                q = phi(m)
                stats["local_evaluated"] += 1
                if phi.satisfies_constraints(m):
                    result_set.insert(q, nb, m)
                if q > best_q:
                    best_q, best_d = q, nb
            if best_d is not None:
                nxt.append(best_d)
        frontier = nxt
        if not frontier:
            break
    return result_set.sorted()


def _rank_of_planted(entries):
    planted_q = [q for _, q, jac, valid in entries if valid and jac >= HIT_JACCARD]
    if not planted_q:
        return np.nan
    best = max(planted_q)
    return 1 + sum(1 for _, q, jac, valid in entries if valid and jac < HIT_JACCARD and q > best)


BUDGETS = [25, 50, 100, 200, 400, 800, 1600, 3200]
SECONDS = [0.1, 0.25, 0.5, 1, 2, 5, 10, 30]

In [ ]:
POST = np.arange(T) >= BASE_LEN


def binary_descriptors(rng, k):
    """Bart's generator: every record gets k attributes ~ Bin(1, 0.5)."""
    return (rng.random((N, k)) < 0.5).astype(np.int8)


def snn_components(D, max_size):
    """Non-convex mini-clusters: connected components of the shared-nearest-neighbour graph, each expanded with its
    members' nearest neighbours (so a ~5-record chain becomes a ~50-record neighbourhood that follows the group)."""
    knn = knn_lists(D)
    edges = [e for e in snn_edges(D) if e[0] >= SNN_MIN]
    if not edges:
        return []
    i = np.array([e[2] for e in edges])
    j = np.array([e[3] for e in edges])
    graph = coo_matrix((np.ones(len(i)), (i, j)), shape=(len(D), len(D)))
    _, labels = connected_components(graph, directed=False)
    comps = pd.Series(np.arange(len(D))).groupby(labels).apply(list)
    comps = [c for c in comps if len(c) >= Z_COMP_MIN]
    expanded = []
    for c in comps:
        members = np.unique(np.concatenate([np.asarray(c), knn[list(c)].ravel()]))
        if len(members) <= max_size:
            expanded.append(tuple(members.tolist()))
    return sorted(set(expanded), key=len, reverse=True)[:U_SEEDS]


def enriched_conditions(mc, A_, overall, z_min=2.0):
    """Conditions shared by >= MAJORITY of the mini-cluster, ranked by a binomial enrichment z-score
    (share in cluster - share overall) * sqrt(n) / sqrt(p (1 - p)). In a large binary space many conditions are shared
    by chance in a small group; the z-score only trusts enrichment that is backed by enough records."""
    sub = A_[list(mc)]
    n = len(mc)
    share1 = sub.mean(axis=0)
    conds = []
    for j in range(A_.shape[1]):
        for v, s, base in ((1, share1[j], overall[j]), (0, 1 - share1[j], 1 - overall[j])):
            if s >= MAJORITY and 0 < base < 1:
                z = (s - base) * np.sqrt(n) / np.sqrt(base * (1 - base))
                if z >= z_min:
                    conds.append((z, (j, v)))
    conds.sort(reverse=True)
    return [c for _, c in conds[:TOP_ENRICHED]]


def local_search_d(A_, phi, starts, result_set, stats, d, topk=LOCAL_TOPK, max_rounds=10):
    frontier = [desc for _, desc, _ in starts[:topk]]
    seen = set(frontier)
    for _ in range(max_rounds):
        nxt = []
        for desc in frontier:
            mask = mask_of(A_, desc)
            best_q, best_d = (phi(mask) if phi.expandable(mask) else -np.inf), None
            neighbours = [tuple(c for c in desc if c != drop) for drop in desc] if len(desc) > 1 else []
            used = {j for j, _ in desc}
            if len(desc) < d:
                for j in range(A_.shape[1]):
                    if j not in used:
                        neighbours += [tuple(sorted(desc + ((j, v),))) for v in np.unique(A_[mask, j])]
            for c in desc:
                rest = tuple(x for x in desc if x != c)
                for v in np.unique(A_[mask_of(A_, rest), c[0]]):
                    if v != c[1]:
                        neighbours.append(tuple(sorted(rest + ((c[0], int(v)),))))
            for nb in neighbours:
                if nb in seen or not nb:
                    continue
                seen.add(nb)
                m = mask_of(A_, nb)
                if not phi.expandable(m):
                    continue
                q = phi(m)
                stats["local_evaluated"] += 1
                if phi.satisfies_constraints(m):
                    result_set.insert(q, nb, m)
                if q > best_q:
                    best_q, best_d = q, nb
            if best_d is not None:
                nxt.append(best_d)
        frontier = nxt
        if not frontier:
            break
    return result_set.sorted()


def _deep_job(spec, excess, level, D_blocks, store_codes, methods):
    rng = np.random.default_rng(spec["seed"])
    k, lits, d = spec["k"], spec["lits"], spec["lits"]
    A_ = binary_descriptors(rng, k)
    attrs = rng.choice(k, lits, replace=False)
    planted_desc = tuple(sorted((int(j), 0) for j in attrs))
    planted = mask_of(A_, planted_desc)
    rows = np.flatnonzero(planted)
    keep = rows[rng.random(len(rows)) >= spec["outliers"]]
    exc = excess.copy()
    exc[keep] += spec["delta"] * POST[None, :] * level[keep, None]
    rel = np.clip(exc / level[:, None], -3, 3)

    t = time.time()
    Db = update_rows(D_blocks, block_means(rel), keep)
    comps = snn_components(Db, max_size=int(MAX_COV * len(level)))
    overall = A_.mean(axis=0)
    comp_conds = [enriched_conditions(mc, A_, overall) for mc in comps]
    seed_time = time.time() - t
    cands = set()
    for conds in comp_conds:
        for size in range(1, min(d, len(conds)) + 1):
            cands.update(tuple(sorted(c)) for c in itertools.combinations(conds, size))
    planted_in = [planted[list(mc)].mean() for mc in comps]
    seed_row = {"components": len(comps), "mean_comp_size": np.mean([len(c) for c in comps]) if comps else 0,
                "best_comp_purity": max(planted_in, default=0.0),
                "comps_mostly_planted": int(sum(p >= 0.5 for p in planted_in)),
                "planted_conds_in_top": max((len(set(cc) & set(planted_desc)) for cc in comp_conds), default=0),
                "candidates": len(cands), "seed_time_s": seed_time}

    out = []
    for method in methods:
        phi = MonetaryQuality(exc, level, store_codes, relative=True, min_cov=MIN_COV_V5)
        phi.track, phi.trace = planted, []
        stats = Counter()
        t0 = time.time()
        offset = 0.0
        if method.startswith("BS"):
            top, stats = beam_search(A_, phi, w=int(method.split("w=")[1]), d=d, prune=True)
        elif method.startswith("BUS→BS pool"):
            offset = seed_time
            votes = Counter(c for conds in comp_conds for c in conds)
            pool = [c for c, _ in votes.most_common(POOL_SIZE)]
            top, stats = beam_search(A_, phi, w=10, d=d, prune=True, pool=pool)
        else:
            offset = seed_time
            top, stats = evaluate_candidates(A_, phi, cands, prune=True)
            if method.endswith("+local"):
                rs = ResultSet(Q)
                for q_, d_, m_ in top:
                    rs.insert(q_, d_, m_)
                top = local_search_d(A_, phi, top, rs, stats, d)
        trace = [(tt - t0 + offset, q_, jac, valid) for tt, q_, jac, valid in phi.trace]
        top_jac = [np.count_nonzero(m & planted) / np.count_nonzero(m | planted) for _, _, m in top]
        ranks = [r + 1 for r, jac in enumerate(top_jac) if jac >= HIT_JACCARD]
        row = {k_: v for k_, v in spec.items()}
        row.update({"method": method, "planted_n": int(planted.sum()), "conditions": 2 * k,
                    "evaluated_hit": any(valid and jac >= HIT_JACCARD for _, _, jac, valid in trace),
                    "rank": ranks[0] if ranks else np.nan, "evaluated": len(trace),
                    "runtime_s": time.time() - t0 + offset, "local_evaluated": stats.get("local_evaluated", 0),
                    **seed_row})
        for b in BUDGETS + [6400, 12800]:
            row[f"rank_at_{b}"] = _rank_of_planted(trace[:b])
        for s in SECONDS:
            row[f"rank_at_{s}s"] = _rank_of_planted([e for e in trace if e[0] <= s])
        out.append(row)
    return out

In [ ]:
def itemset_support(baskets, d):
    """Support of every combination (up to d conditions) = number of baskets containing all of it.
    Baskets are small (<= TOP_ENRICHED conditions), so counting all sub-combinations per basket is exact and cheap;
    this is what Apriori computes level by level for large baskets."""
    support = Counter()
    for basket in baskets:
        items = sorted(set(basket))
        for size in range(1, min(d, len(items)) + 1):
            support.update(itertools.combinations(items, size))
    return support


def guided_beam(A_, phi, starts, result_set, stats, d, support, width=GUIDED_WIDTH, max_rounds=10):
    """Small-beam local search whose add / swap moves are restricted to conditions that co-occur (pair support >= 1)
    with a condition already in the description, best-supported first."""
    pair = {k: v for k, v in support.items() if len(k) == 2}
    partners = {}
    for (a, b), s in pair.items():
        partners.setdefault(a, Counter())[b] += s
        partners.setdefault(b, Counter())[a] += s

    def suggestions(desc):
        score = Counter()
        for c in desc:
            score.update(partners.get(c, {}))
        used = {j for j, _ in desc}
        return [c for c, _ in score.most_common() if c[0] not in used][:GUIDED_ADD_TOP]

    def quality(desc):
        m = mask_of(A_, desc)
        if not phi.expandable(m):
            return None
        q = phi(m)
        stats["local_evaluated"] += 1
        if phi.satisfies_constraints(m):
            result_set.insert(q, desc, m)
        return q

    beam = [(q, desc) for q, desc, _ in starts[:width]]
    seen = {desc for _, desc in beam}
    for _ in range(max_rounds):
        cands = []
        for _, desc in beam:
            neighbours = [tuple(c for c in desc if c != drop) for drop in desc] if len(desc) > 1 else []
            for c in suggestions(desc):
                if len(desc) < d:
                    neighbours.append(tuple(sorted(desc + (c,))))
                for old in desc:
                    neighbours.append(tuple(sorted(tuple(x for x in desc if x != old) + (c,))))
            for nb in neighbours:
                if nb and nb not in seen:
                    seen.add(nb)
                    q = quality(nb)
                    if q is not None:
                        cands.append((q, nb))
        new_beam = sorted(beam + cands, key=lambda x: -x[0])[:width]
        if {x for _, x in new_beam} == {x for _, x in beam}:
            break
        beam = new_beam
    return result_set.sorted()


def _deep_job_v6(spec, excess, level, D_blocks, store_codes, methods):
    rng = np.random.default_rng(spec["seed"])
    k, lits, d = spec["k"], spec["lits"], spec["lits"]
    A_ = binary_descriptors(rng, k)
    attrs = rng.choice(k, lits, replace=False)
    planted_desc = tuple(sorted((int(j), 0) for j in attrs))
    planted = mask_of(A_, planted_desc)
    rows = np.flatnonzero(planted)
    keep = rows[rng.random(len(rows)) >= spec["outliers"]]
    exc = excess.copy()
    exc[keep] += spec["delta"] * POST[None, :] * level[keep, None]
    rel = np.clip(exc / level[:, None], -3, 3)

    t = time.time()
    Db = update_rows(D_blocks, block_means(rel), keep)
    comps = snn_components(Db, max_size=int(MAX_COV * len(level)))
    overall = A_.mean(axis=0)
    comp_conds = [enriched_conditions(mc, A_, overall) for mc in comps]
    seed_time = time.time() - t
    cands = set()
    for conds in comp_conds:
        for size in range(1, min(d, len(conds)) + 1):
            cands.update(tuple(sorted(c)) for c in itertools.combinations(conds, size))
    t = time.time()
    support = itemset_support(comp_conds, d)
    frequent = sorted((s for s, n in support.items() if n >= MIN_SUPPORT), key=lambda s: (-support[s], -len(s)))
    itemset_time = time.time() - t
    planted_in = [planted[list(mc)].mean() for mc in comps]
    seed_row = {"components": len(comps), "mean_comp_size": np.mean([len(c) for c in comps]) if comps else 0,
                "best_comp_purity": max(planted_in, default=0.0),
                "comps_mostly_planted": int(sum(p >= 0.5 for p in planted_in)),
                "planted_conds_in_top": max((len(set(cc) & set(planted_desc)) for cc in comp_conds), default=0),
                "planted_support": support.get(planted_desc, 0), "frequent_itemsets": len(frequent),
                "candidates": len(cands), "seed_time_s": seed_time}

    out = []
    for method in methods:
        phi = MonetaryQuality(exc, level, store_codes, relative=True, min_cov=MIN_COV_V5)
        phi.track, phi.trace = planted, []
        stats = Counter()
        t0 = time.time()
        offset = 0.0
        if method.startswith("BS"):
            top, stats = beam_search(A_, phi, w=int(method.split("w=")[1]), d=d, prune=True)
        elif method.startswith("BUS→BS pool"):
            offset = seed_time
            votes = Counter(c for conds in comp_conds for c in conds)
            top, stats = beam_search(A_, phi, w=10, d=d, prune=True, pool=[c for c, _ in votes.most_common(POOL_SIZE)])
        elif method.startswith("BUS itemsets"):
            offset = seed_time + itemset_time
            top, stats = evaluate_candidates(A_, phi, set(frequent), prune=True)
            if method.endswith("+guided beam"):
                rs = ResultSet(Q)
                for q_, d_, m_ in top:
                    rs.insert(q_, d_, m_)
                top = guided_beam(A_, phi, top, rs, stats, d, support)
        else:
            offset = seed_time
            top, stats = evaluate_candidates(A_, phi, cands, prune=True)
            if method.endswith("+local"):
                rs = ResultSet(Q)
                for q_, d_, m_ in top:
                    rs.insert(q_, d_, m_)
                top = local_search_d(A_, phi, top, rs, stats, d)
        trace = [(tt - t0 + offset, q_, jac, valid) for tt, q_, jac, valid in phi.trace]
        top_jac = [np.count_nonzero(m & planted) / np.count_nonzero(m | planted) for _, _, m in top]
        ranks = [r + 1 for r, jac in enumerate(top_jac) if jac >= HIT_JACCARD]
        row = dict(spec)
        row.update({"method": method, "planted_n": int(planted.sum()), "conditions": 2 * k,
                    "evaluated_hit": any(valid and jac >= HIT_JACCARD for _, _, jac, valid in trace),
                    "rank": ranks[0] if ranks else np.nan, "evaluated": len(trace),
                    "runtime_s": time.time() - t0 + offset, "local_evaluated": stats.get("local_evaluated", 0),
                    **seed_row})
        for b in BUDGETS + [6400, 12800]:
            row[f"rank_at_{b}"] = _rank_of_planted(trace[:b])
        for s in SECONDS:
            row[f"rank_at_{s}s"] = _rank_of_planted([e for e in trace if e[0] <= s])
        out.append(row)
    return out

In [ ]:
# ================= binary study: settings =================
K_ATTRS = [20, 400] if QUICK else [20, 50, 100, 400]   # binary attributes (2 conditions each)
LITS_B = [2, 10] if QUICK else [2, 4, 6, 8, 10]         # conditions in the planted (true) description
GRID = [(8, 0.0)] if QUICK else [(8, 0.0), (8, 0.2), (16, 0.0), (16, 0.2)]   # (snr, outliers)
REPS_B = 1 if QUICK else 10
P_RANGE = (0.5, 0.9)              # share of records with value 1, drawn per attribute (uneven binary attributes)
COVERAGE_RANGE = (0.01, 0.15)     # accepted coverage of a planted subgroup (= the min / max coverage constraints)
MIN_COV_B = 0.01                  # minimum coverage constraint for every method
TOP_ENRICHED = 8                  # SNN BUS: conditions per mini-cluster (raised to the depth if the depth is larger)
BART_MIN_TIME = 60                # Bart's exact BUS gets as much time as BS w=50 took on the same task, at least this
                                  # many seconds (Engelen §4.3.4: time budget); unfinished runs are reported
POOL_W = 10                       # beam width of both BUS→BS pool ensembles
METHODS_B = ['BS w=10', 'BS w=50', 'BUS', 'BUS +local', 'BUS-α', 'BUS→BS pool', 'BUS snn', 'BUS snn +local', 'BUS→BS pool snn']
RESULTS_DIR = f"results_binary_quick/{DATASET}" if QUICK else f"results_binary/{DATASET}"
os.makedirs(RESULTS_DIR, exist_ok=True)
N_JOBS = min(N_JOBS, 7)
BATCH = 2 * N_JOBS
print(DATASET, "| methods:", METHODS_B)

In [ ]:
# ================= binary study: generator, seeds, methods =================
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components


def binary_uneven(rng, k):
    """k binary attributes; attribute j is 1 for a share p_j ~ U(P_RANGE) of the records (Bart's generator has p = 0.5)."""
    p = rng.uniform(P_RANGE[0], P_RANGE[1], size=k)
    return (rng.random((N, k)) < p).astype(np.int8)


def value_freqs(A_):
    return [np.bincount(A_[:, j], minlength=2) / len(A_) for j in range(A_.shape[1])]


def n_stores_of(mask):
    return len(np.unique(store_codes[mask]))


def sample_true_description(rng, A_, freqs, lits, tries=5000):
    """lits random attributes, each with a value drawn in proportion to its frequency; resampled until the subgroup
    covers COVERAGE_RANGE of the records and at least MIN_STORES stores."""
    for _ in range(tries):
        attrs = rng.choice(A_.shape[1], lits, replace=False)
        desc = tuple(sorted((int(j), int(rng.choice(2, p=freqs[j]))) for j in attrs))
        mask = mask_of(A_, desc)
        if COVERAGE_RANGE[0] <= mask.mean() <= COVERAGE_RANGE[1] and n_stores_of(mask) >= MIN_STORES:
            return desc, mask
    return None, None


def block_means_n(rel, n_blocks=N_BLOCKS):
    post = rel[:, BASE_LEN:]
    k = max(1, post.shape[1] // n_blocks)
    m = (post.shape[1] // k) * k
    return post[:, :m].reshape(len(rel), -1, k).mean(axis=2)


def pairwise_sqeuclidean(X, block=512):
    """Full N x N squared Euclidean distance matrix (Engelen eq. 2.15), row blocks in parallel."""
    starts = list(range(0, len(X), block))
    parts = Parallel(n_jobs=N_JOBS)(delayed(cdist)(X[s:s + block], X, "sqeuclidean") for s in starts)
    D = np.empty((len(X), len(X)), dtype=np.float32 if len(X) > 6000 else np.float64)
    for s, part in zip(starts, parts):
        D[s:s + len(part)] = part
    return D


def update_rows_sq(D, X, rows):
    D = D.copy()
    if len(rows):
        new = cdist(X[rows], X, "sqeuclidean").astype(D.dtype)
        D[rows, :] = new
        D[:, rows] = new.T
    return D


# ---------------- Bart's BUS (Engelen Alg. 2/3), exact, with a time budget
def bart_exact(pairs, A_, d, phi, time_limit):
    """Every combination (up to depth d) of the conditions a pair shares, level by level. A combination is skipped
    only if a sub-description failed the coverage constraint or its optimistic estimate was below the current q-th
    quality: both rules are lossless, so the top-q equals that of the full Alg. 3 if the run finishes."""
    spaces = [[(int(j), int(v)) for j, v in shared_conditions(list(pr), A_)] for pr in pairs]
    rs, stats, t_end = ResultSet(Q), Counter(), time.time() + time_limit
    fronts, prev_alive = [[()] for _ in spaces], {()}
    for size in range(1, d + 1):
        seen, cur_alive, new_fronts = {}, set(), []
        for S, front in zip(spaces, fronts):
            nf = []
            for I in front:
                for k in range(I[-1] + 1 if I else 0, len(S)):
                    J = I + (k,)
                    desc = tuple(S[i] for i in J)
                    if desc not in seen:
                        ok = size == 1 or all(desc[:m] + desc[m + 1:] in prev_alive for m in range(size))
                        if ok:
                            mask = mask_of(A_, desc)
                            if not phi.expandable(mask):
                                ok = False
                            else:
                                quality = phi(mask)
                                if phi.satisfies_constraints(mask):
                                    rs.insert(quality, desc, mask)
                                if oe_below_threshold(phi, mask, rs):
                                    ok = False
                        seen[desc] = ok
                        if ok:
                            cur_alive.add(desc)
                        if time.time() > t_end:
                            return rs.sorted(), stats, False, spaces
                    if seen[desc]:
                        nf.append(J)
            new_fronts.append(nf)
        stats["levels_done"] = size
        fronts, prev_alive = new_fronts, cur_alive
    return rs.sorted(), stats, True, spaces


# ---------------- fixed BUS: SNN components on the scale-free, time-blocked target
def snn_components_exp(D, max_size, k=KNN, snn_min=SNN_MIN, u=U_SEEDS):
    knn = knn_lists(D, k=k)
    sets = [set(r) for r in knn.tolist()]
    edges = [(i, j) for i, row in enumerate(knn.tolist()) for j in row
             if j > i and i in sets[j] and len(sets[i] & sets[j]) >= snn_min]
    if not edges:
        return []
    i = np.array([e[0] for e in edges])
    j = np.array([e[1] for e in edges])
    _, labels = connected_components(coo_matrix((np.ones(len(i)), (i, j)), shape=(len(D), len(D))), directed=False)
    comps = [c for c in pd.Series(np.arange(len(D))).groupby(labels).apply(list) if len(c) >= Z_COMP_MIN]
    out = []
    for c in comps:
        members = np.unique(np.concatenate([np.asarray(c), knn[list(c)].ravel()]))
        if len(members) <= max_size:
            out.append(tuple(members.tolist()))
    return sorted(set(out), key=len, reverse=True)[:u]


def enriched(mc, A_, freqs, top, z_min=2.0):
    """Conditions shared by >= MAJORITY of the mini-cluster, ranked by binomial enrichment z (z >= z_min), at most top."""
    sub = A_[list(mc)]
    n = len(mc)
    conds = []
    for j in range(A_.shape[1]):
        ones = int(sub[:, j].sum())
        for v, c in ((1, ones), (0, n - ones)):
            s, base = c / n, freqs[j][v]
            if s >= MAJORITY and 0 < base < 1:
                z = (s - base) * np.sqrt(n) / np.sqrt(base * (1 - base))
                if z >= z_min:
                    conds.append((z, (j, v)))
    conds.sort(reverse=True)
    return [c for _, c in conds[:top]]


def parse_method(name):
    if name.startswith("BS"):
        return {"kind": "bs", "w": int(name.split("w=")[1]), "local": False}
    seeds = "snn" if "snn" in name else ("alpha" if ("α" in name or "pool" in name) else "closest")
    kind = "pool" if "pool" in name else "bus"
    return {"kind": kind, "seeds": seeds, "local": name.endswith("+local"),
            "base": name.replace(" +local", "") if name.endswith("+local") else None}


METHOD_SPECS = {m: parse_method(m) for m in METHODS_B}
BUDGETS_B = [100, 200, 400, 800, 1600, 3200, 6400, 12800, 25600, 51200]
SECONDS_B = [1, 2, 5, 10, 30, 60, 120]


def _binary_job(spec, excess, level, store_codes_, D_blocks, D_bart):
    rng = np.random.default_rng(spec["seed"])
    lits = spec["lits"]
    d = lits                                       # every method searches to the true depth
    A_ = binary_uneven(rng, spec["k"])
    freqs = value_freqs(A_)
    planted_desc, planted = sample_true_description(rng, A_, freqs, lits)
    if planted_desc is None:
        return []
    rows_ = np.flatnonzero(planted)
    keep = rows_[rng.random(len(rows_)) >= spec["outliers"]]
    exc = excess.copy()
    exc[keep] += spec["delta"] * POST[None, :] * level[keep, None]
    rel = np.clip(exc / level[:, None], -3, 3)
    max_size = int(MAX_COV * len(level))
    pset = set(planted_desc)

    cache = {}

    def seeds(kind):
        """Mini-clusters + the time to build them (distance update included)."""
        if kind not in cache:
            t = time.time()
            if kind == "snn":
                D = update_rows(D_blocks, block_means_n(rel), keep)
                mcs = snn_components_exp(D, max_size)
            else:
                if "D_bart" not in cache:
                    cache["D_bart"] = update_rows_sq(D_bart, exc, keep)
                mcs = top_pairs(cache["D_bart"], U_SEEDS, kind)
            cache[kind] = (mcs, time.time() - t)
        return cache[kind]

    def make_row(name, phi, top, t0, offset, info):
        trace = [(tt - t0 + offset, q_, jac, valid) for tt, q_, jac, valid in phi.trace]
        top_jac = [np.count_nonzero(m & planted) / np.count_nonzero(m | planted) for _, _, m in top]
        ranks = [r + 1 for r, jac in enumerate(top_jac) if jac >= HIT_JACCARD]
        row = {k_: v_ for k_, v_ in spec.items()}
        row.update({"method": name, "depth_searched": d, "planted_n": int(planted.sum()), "coverage": planted.mean(),
                    "evaluated_hit": any(valid and jac >= HIT_JACCARD for _, _, jac, valid in trace),
                    "rank": ranks[0] if ranks else np.nan, "evaluated": len(trace),
                    "runtime_s": time.time() - t0 + offset, "seed_time_s": offset, **info})
        for b in BUDGETS_B:
            row[f"rank_at_{b}"] = _rank_of_planted(trace[:b])
        for s_ in SECONDS_B:
            row[f"rank_at_{s_}s"] = _rank_of_planted([e for e in trace if e[0] <= s_])
        return row

    out, state, bs50 = [], {}, {"runtime": 0.0}
    for name, s in METHOD_SPECS.items():
        if s["local"]:                              # continue from the base method's run (same quality object)
            if s["base"] not in state:
                raise ValueError(f"{name} must come right after {s['base']}")
            phi, top, t0, offset, info = state[s["base"]]
            rs = ResultSet(Q)
            for q_, d_, m_ in top:
                rs.insert(q_, d_, m_)
            stats = Counter()
            top = local_search_d(A_, phi, top, rs, stats, d)
            out.append(make_row(name, phi, top, t0, offset, dict(info, local_evaluated=stats["local_evaluated"])))
            continue
        phi = MonetaryQuality(exc, level, store_codes_, relative=True, min_cov=MIN_COV_B)
        phi.track, phi.trace = planted, []
        info, offset = {}, 0.0
        if s["kind"] == "bs":
            t0 = time.time()
            top, _ = beam_search(A_, phi, w=s["w"], d=d, prune=True)
            if s["w"] == 50:
                bs50["runtime"] = time.time() - t0
        else:
            mcs, offset = seeds(s["seeds"])
            t0 = time.time()
            if s["seeds"] == "snn":
                baskets = [enriched(mc, A_, freqs, max(TOP_ENRICHED, d)) for mc in mcs]
            else:
                baskets = [[(int(j), int(v)) for j, v in shared_conditions(list(mc), A_)] for mc in mcs]
            info = {"mini_clusters": len(mcs), "mean_mc_size": float(np.mean([len(m) for m in mcs])) if mcs else 0.0,
                    "best_mc_purity": max((planted[list(m)].mean() for m in mcs), default=0.0),
                    "pair_size_pct": float(np.mean([LEVEL_PCT[list(m)].mean() for m in mcs])) if mcs else np.nan,
                    "planted_conds_in_basket": max((len(set(b) & pset) for b in baskets), default=0),
                    "planted_in_space": any(pset <= set(b) for b in baskets),
                    "basket_size": float(np.mean([len(b) for b in baskets])) if baskets else 0.0}
            if s["kind"] == "pool":
                votes = Counter(c for b in baskets for c in b)
                top, _ = beam_search(A_, phi, w=POOL_W, d=d, prune=True,
                                     pool=[c for c, _ in votes.most_common(POOL_SIZE)])
            elif s["seeds"] == "snn":
                cands = set()
                for b in baskets:
                    for size in range(1, min(d, len(b)) + 1):
                        cands.update(tuple(sorted(c)) for c in itertools.combinations(b, size))
                info["candidates"] = len(cands)
                top, _ = evaluate_candidates(A_, phi, cands, prune=True)
            else:
                budget = max(BART_MIN_TIME, bs50["runtime"])
                top, st, finished, _ = bart_exact(mcs, A_, d, phi, budget)
                info.update(finished=finished, levels_done=st["levels_done"], time_budget_s=budget)
        state[name] = (phi, top, t0, offset, info)
        out.append(make_row(name, phi, top, t0, offset, info))
    return out

In [ ]:
# ---- calibration (deterministic): null spread of random true-subgroup-like descriptions, per depth
rng = np.random.default_rng(SEED + 11)
SIGMA_LITS = {}
for lits in LITS_B:
    rates = []
    while len(rates) < 200:
        A_ = binary_uneven(rng, 100)
        desc, mask = sample_true_description(rng, A_, value_freqs(A_), lits)
        if desc is not None:
            rates.append(excess[mask][:, POST].sum() / (level[mask].sum() * POST.sum()))
    SIGMA_LITS[lits] = float(np.std(rates))
print("null spread per depth:", {k: round(v, 4) for k, v in SIGMA_LITS.items()})

LEVEL_PCT = pd.Series(level).rank(pct=True).to_numpy()      # size of a series (0 = smallest, 1 = largest)
D_blocks = pairwise_cityblock(block_means_n(rel_dev))      # fixed BUS: scale-free, time-blocked target
D_bart = pairwise_sqeuclidean(excess)                      # Bart's BUS: squared Euclidean on the target as-is

specs = []
for k in K_ATTRS:
    for lits in LITS_B:
        for snr, outliers in GRID:
            for rep in range(REPS_B):
                specs.append({"dataset": DATASET, "k": k, "lits": lits, "snr": snr, "outliers": outliers, "rep": rep,
                              "delta": snr * SIGMA_LITS[lits], "seed": int(rng.integers(1 << 31))})

out_csv, progress = f"{RESULTS_DIR}/tasks.csv", f"{RESULTS_DIR}/progress.txt"
skipped_file = f"{RESULTS_DIR}/skipped_seeds.txt"
done = set(pd.read_csv(out_csv)["seed"]) if os.path.exists(out_csv) else set()
skipped = set(int(x) for x in open(skipped_file).read().split()) if os.path.exists(skipped_file) else set()
todo = sorted([s for s in specs if s["seed"] not in done and s["seed"] not in skipped], key=lambda s: (s["rep"], s["lits"], s["k"]))
print(f"{DATASET}: {len(specs)} tasks x {len(METHODS_B)} methods; {len(done) + len(skipped)} already done, {len(todo)} to run")
start = time.time()
for b in range(0, len(todo), BATCH):
    batch = todo[b:b + BATCH]
    parts = Parallel(n_jobs=N_JOBS)(delayed(_binary_job)(s, excess, level, store_codes, D_blocks, D_bart) for s in batch)
    rows = [r for p in parts for r in p]
    if rows:
        new = pd.DataFrame(rows)
        if os.path.exists(out_csv):            # same columns, same order as the file
            cols = list(pd.read_csv(out_csv, nrows=0).columns)
            extra = [c for c in new.columns if c not in cols]
            assert not extra, f"new columns {extra} not in {out_csv}"
            new.reindex(columns=cols).to_csv(out_csv, mode="a", header=False, index=False)
        else:
            new.to_csv(out_csv, index=False)
    with open(skipped_file, "a") as f:
        for s, p in zip(batch, parts):
            if not p:
                f.write(f"{s['seed']}\n")
    elapsed = time.time() - start
    left = elapsed / (b + len(batch)) * (len(todo) - b - len(batch))
    with open(progress, "w") as f:
        f.write(f"{DATASET}: {len(done) + len(skipped) + b + len(batch)} / {len(specs)} tasks, "
                f"{elapsed / 60:.1f} min elapsed, ~{left / 60:.1f} min left\n")
print(f"done in {time.time() - start:.0f}s")

In [ ]:
# ---- results: every method vs BS w=10 and vs BUS snn on the same tasks (exact McNemar test)
from scipy.stats import binomtest

res = pd.read_csv(f"{RESULTS_DIR}/tasks.csv")
res["found"] = res["rank"] <= Q
methods = [m for m in METHODS_B if m in set(res.method)]
key = ["k", "lits", "snr", "outliers", "rep"]
print(f"{DATASET}: {res.seed.nunique()} tasks built (of {len(specs)}; the rest had no valid true subgroup)")
print("Average per task")
display(res.groupby("method")[["found", "evaluated_hit", "evaluated", "runtime_s"]].mean().reindex(methods).round(3))
print("Share found, by depth (rows) and method (columns)")
display(res.pivot_table(index="lits", columns="method", values="found")[methods].round(3))
print("Share found, by number of binary attributes k")
display(res.pivot_table(index="k", columns="method", values="found")[methods].round(3))
if "finished" in res:
    print("Bart's exact BUS: share of runs finished within the time limit, by k and depth")
    display(res[res.method.isin(["BUS", "BUS-α"])].pivot_table(index="k", columns="lits", values="finished").round(2))

w = res.pivot_table(index=key, columns="method", values="found").astype(bool)
families = [([("all", w)], 0.05),
            ([(f"lits={l}", w.xs(l, level="lits", drop_level=False)) for l in sorted(res.lits.unique())], 0.05 / res.lits.nunique()),
            ([(f"k={k}", w.xs(k, level="k", drop_level=False)) for k in sorted(res.k.unique())], 0.05 / res.k.nunique())]
rows = []
for ref in ["BS w=10", "BUS snn"]:
    for groups, alpha in families:
        for g, sub in groups:
            for m in [x for x in methods if x != ref]:
                wins, losses = int((sub[m] & ~sub[ref]).sum()), int((~sub[m] & sub[ref]).sum())
                p = binomtest(wins, wins + losses).pvalue if wins + losses else 1.0
                verdict = (f"better than {ref}" if wins > losses else f"worse than {ref}") if p < alpha else \
                          ("suggestive (not after correction)" if p < 0.05 else "no significant difference")
                rows.append({"vs": ref, "group": g, "method": m, "tasks": len(sub), "found": sub[m].mean(),
                             "found_ref": sub[ref].mean(), "wins": wins, "losses": losses, "p": p, "alpha": alpha,
                             "verdict": verdict})
paired = pd.DataFrame(rows)
paired.to_csv(f"{RESULTS_DIR}/paired.csv", index=False)
print("Paired comparison (wins = only this method found it, losses = only the reference found it)")
display(paired[paired.vs == "BS w=10"].round(4))
display(paired[paired.vs == "BUS snn"].round(4))